# Attention Visualization for Bbox Coordinate Generation
Visualize where Qwen3-VL attends when generating each coordinate (x1, y1, x2, y2).

In [ ]:
import json, re, torch, yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

DATA_DIR = Path('/efs/user_folders/dnshalam/datasets/lvis')

with open('configs/lora_config.yaml') as f:
    config = yaml.safe_load(f)

model_name = config['model_name']
processor = AutoProcessor.from_pretrained(model_name)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map='auto',
    attn_implementation='eager',
)
model.eval()
print(f'Loaded {model_name}')

In [ ]:
# Load validation data
with open(DATA_DIR / 'lvis_validation.json') as f:
    data = json.load(f)
print(f'{len(data)} samples')

In [ ]:
# ── Helper functions ──

def get_image_token_mask(input_ids):
    image_token_id = processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')
    return (input_ids[0] == image_token_id).cpu().numpy()

def generate_with_attention(inputs, max_new_tokens=128):
    input_ids = inputs['input_ids'].clone()
    records = []
    for _ in range(max_new_tokens):
        with torch.no_grad():
            out = model(**{k: v for k, v in inputs.items() if k != 'input_ids'},
                        input_ids=input_ids, output_attentions=True)
        next_tok = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        per_layer = [la[0, :, -1, :].cpu().float().numpy() for la in out.attentions]
        records.append({'token_id': next_tok.item(), 'per_layer': per_layer})
        input_ids = torch.cat([input_ids, next_tok], dim=-1)
        if next_tok.item() == model.config.eos_token_id:
            break
    return input_ids, records

def find_coord_steps(records):
    tokens = [processor.tokenizer.decode(r['token_id']) for r in records]
    text = ''.join(tokens)
    m = re.search(r'"bbox_2d"\s*:\s*\[(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\]', text)
    if not m: return None, tokens, text
    steps = {}
    pos = 0
    for i, t in enumerate(tokens):
        for j, name in enumerate(['x1','y1','x2','y2']):
            if name not in steps and pos <= m.start(j+1) < pos + len(t):
                steps[name] = i
        pos += len(t)
    return steps, tokens, text

def attn_breakdown(attn_vec, img_mask, n_input):
    total = attn_vec.sum()
    if total == 0: return {'image':0,'text':0,'generated':0}
    return {
        'image': float(attn_vec[:n_input][img_mask[:n_input]].sum()/total),
        'text': float(attn_vec[:n_input][~img_mask[:n_input]].sum()/total),
        'generated': float(attn_vec[n_input:].sum()/total),
    }

def spatial_heatmap(attn_vec, img_mask, w, h):
    ia = attn_vec[:len(img_mask)][img_mask[:len(attn_vec)]]
    n = ia.shape[0]
    if n == 0: return np.zeros((h, w))
    side = int(np.ceil(np.sqrt(n)))
    ia = np.pad(ia, (0, side*side - n))
    return np.array(Image.fromarray(ia.reshape(side,side)).resize((w,h), Image.BILINEAR))

In [ ]:
# ── Run inference on a sample ──
SAMPLE_IDX = 0  # <-- change this to explore different samples

item = data[SAMPLE_IDX]
prompt_text = re.sub(r'<img>.*?</img>\n', '', item['conversations'][0]['value'])
gt_match = re.search(r'<box>\((\d+),(\d+)\),\((\d+),(\d+)\)</box>', item['conversations'][1]['value'])
gt_coords = [int(gt_match.group(i)) for i in range(1,5)] if gt_match else None

image = Image.open(item['image']).convert('RGB')
messages = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt_text}]}]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[image], return_tensors='pt').to(model.device)

n_input = inputs['input_ids'].shape[1]
img_mask = get_image_token_mask(inputs['input_ids'])
print(f'Prompt: {prompt_text}')
print(f'Input tokens: {n_input}, Image tokens: {img_mask.sum()}')

output_ids, attn_records = generate_with_attention(inputs)
coord_steps, tokens, gen_text = find_coord_steps(attn_records)
print(f'Generated: {gen_text[:200]}')
print(f'Coordinate steps: {coord_steps}')

num_layers = len(attn_records[0]['per_layer'])
num_heads = attn_records[0]['per_layer'][0].shape[0]
print(f'Model: {num_layers} layers, {num_heads} heads')

## 1. Attention Breakdown Across All Layers (head-averaged)

In [ ]:
if coord_steps:
    coord_names = ['x1','y1','x2','y2']
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    for col, name in enumerate(coord_names):
        if name not in coord_steps: continue
        step = coord_steps[name]
        img_ratios, txt_ratios, gen_ratios = [], [], []
        for li in range(num_layers):
            attn = attn_records[step]['per_layer'][li].mean(axis=0)
            bd = attn_breakdown(attn, img_mask, n_input)
            img_ratios.append(bd['image'])
            txt_ratios.append(bd['text'])
            gen_ratios.append(bd['generated'])
        x = range(num_layers)
        axes[col].stackplot(x, img_ratios, txt_ratios, gen_ratios,
                           labels=['Image','Text','Generated'],
                           colors=['#2196F3','#FF9800','#4CAF50'], alpha=0.8)
        axes[col].set_title(name, fontsize=14, fontweight='bold')
        axes[col].set_xlabel('Layer')
        axes[col].set_ylim(0, 1)
        if col == 0:
            axes[col].set_ylabel('Attention ratio')
            axes[col].legend(loc='upper right', fontsize=8)
    fig.suptitle(f'Attention Breakdown Across Layers\n{prompt_text}', fontsize=12)
    plt.tight_layout()
    plt.show()

## 2. Spatial Heatmaps Across Layers

In [ ]:
if coord_steps:
    sample_layers = np.linspace(0, num_layers-1, 6, dtype=int)
    fig, axes = plt.subplots(len(sample_layers), 4, figsize=(16, 3*len(sample_layers)))
    for col, name in enumerate(coord_names):
        if name not in coord_steps: continue
        step = coord_steps[name]
        for row, li in enumerate(sample_layers):
            attn = attn_records[step]['per_layer'][li].mean(axis=0)
            hm = spatial_heatmap(attn, img_mask, image.width, image.height)
            ax = axes[row, col]
            ax.imshow(image)
            ax.imshow(hm, alpha=0.5, cmap='hot')
            if gt_coords:
                gx1,gy1,gx2,gy2 = [c/1000 for c in gt_coords]
                ax.add_patch(patches.Rectangle((gx1*image.width,gy1*image.height),
                    (gx2-gx1)*image.width,(gy2-gy1)*image.height,
                    lw=2,ec='lime',fc='none',ls='--'))
            ax.axis('off')
            if row == 0: ax.set_title(name, fontsize=13, fontweight='bold')
            if col == 0: ax.set_ylabel(f'Layer {li}', fontsize=10); ax.yaxis.set_visible(True)
    fig.suptitle(f'Spatial Attention Across Layers\n{prompt_text}', fontsize=12)
    plt.tight_layout()
    plt.show()

## 3. Per-Head Analysis (single layer)
Change `LAYER_IDX` to explore different layers.

In [ ]:
LAYER_IDX = -1  # <-- last layer by default, change to explore
li = LAYER_IDX if LAYER_IDX >= 0 else num_layers + LAYER_IDX

if coord_steps:
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    for col, name in enumerate(coord_names):
        if name not in coord_steps: continue
        step = coord_steps[name]
        layer_attn = attn_records[step]['per_layer'][li]  # (num_heads, seq_len)
        img_r, txt_r, gen_r = [], [], []
        for hi in range(num_heads):
            bd = attn_breakdown(layer_attn[hi], img_mask, n_input)
            img_r.append(bd['image']); txt_r.append(bd['text']); gen_r.append(bd['generated'])
        x = range(num_heads)
        axes[col].bar(x, img_r, label='Image', color='#2196F3', alpha=0.8)
        axes[col].bar(x, txt_r, bottom=img_r, label='Text', color='#FF9800', alpha=0.8)
        axes[col].bar(x, gen_r, bottom=[i+t for i,t in zip(img_r,txt_r)], label='Generated', color='#4CAF50', alpha=0.8)
        axes[col].set_title(name, fontsize=14, fontweight='bold')
        axes[col].set_xlabel('Head')
        axes[col].set_ylim(0, 1)
        if col == 0:
            axes[col].set_ylabel('Attention ratio')
            axes[col].legend(fontsize=8)
    fig.suptitle(f'Per-Head Attention Breakdown (Layer {li})\n{prompt_text}', fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# Spatial heatmaps for sampled heads
if coord_steps:
    sample_heads = np.linspace(0, num_heads-1, min(8, num_heads), dtype=int)
    fig, axes = plt.subplots(len(sample_heads), 4, figsize=(16, 3*len(sample_heads)))
    if len(sample_heads) == 1: axes = axes[np.newaxis,:]
    for col, name in enumerate(coord_names):
        if name not in coord_steps: continue
        step = coord_steps[name]
        layer_attn = attn_records[step]['per_layer'][li]
        for row, hi in enumerate(sample_heads):
            hm = spatial_heatmap(layer_attn[hi], img_mask, image.width, image.height)
            ax = axes[row, col]
            ax.imshow(image)
            ax.imshow(hm, alpha=0.5, cmap='hot')
            if gt_coords:
                gx1,gy1,gx2,gy2 = [c/1000 for c in gt_coords]
                ax.add_patch(patches.Rectangle((gx1*image.width,gy1*image.height),
                    (gx2-gx1)*image.width,(gy2-gy1)*image.height,
                    lw=2,ec='lime',fc='none',ls='--'))
            ax.axis('off')
            if row == 0: ax.set_title(name, fontsize=13, fontweight='bold')
            if col == 0: ax.set_ylabel(f'Head {hi}', fontsize=10); ax.yaxis.set_visible(True)
    fig.suptitle(f'Spatial Attention per Head (Layer {li})\n{prompt_text}', fontsize=12)
    plt.tight_layout()
    plt.show()